In [8]:
# Summariation middleware

import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage

# message based summarization middleware
agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(
        model="google_genai:gemini-2.5-flash-lite",
        trigger=("messages",10),
        keep=("messages",4)
    )]
)

In [6]:
from langchain_core.messages.human import HumanMessage
# run with thread id
config = {"configurable":{"thread_id": "test-1"}}

# alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?"
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]},config=config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}") 

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='f22eab38-1b9b-414e-b351-092301f97f04'), AIMessage(content='2 + 2 = **4**', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019b88da-2825-79f0-b3d2-8ddc653b3364-0', usage_metadata={'input_tokens': 8, 'output_tokens': 8, 'total_tokens': 16, 'input_token_details': {'cache_read': 0}})]}
Messages: 2
Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='f22eab38-1b9b-414e-b351-092301f97f04'), AIMessage(content='2 + 2 = **4**', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019b88da-2825-79f0-b3d2-8ddc653b3364-0', usage_metadata={'input_tokens': 8, 'output_tokens': 8, 'total_tokens':

In [20]:
# token size 
from langchain_core.tools import tool

@tool
def search_hotel(city:str) -> str:
    """Search hotels = returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""



# message based summarization middleware
agent_with_tool = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[search_hotel],
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(
        model="groq:openai/gpt-oss-120b",
        trigger=("tokens",550),
        keep=("tokens",200)
    )]
)

config_for_token = {"configurable":{"thread_id": "test-2"}}

# token counter (approximate
def token_counter(messages):
    return sum([len(message.content) for message in messages]) // 4

In [21]:
# run test
cities = ["New York", "Los Angeles", "Tokyo", "Houston", "Singapore"]

for city in cities:
    print(f"\nHotel search for {city}")
    response = agent_with_tool.invoke(
        {"messages":[HumanMessage(content=f"Find hotels in {city}")]},
        config=config_for_token)

    token = token_counter(response['messages'])
    print(f"{city}: ~{token} tokens, {len(response['messages'])} messages")
    print(f"{response['messages']}")


Hotel search for New York
New York: ~330 tokens, 4 messages
[HumanMessage(content='Find hotels in New York', additional_kwargs={}, response_metadata={}, id='55d96c46-f4be-426a-91d0-61df7c7280b8'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to find hotels in New York. We have a function search_hotel that takes a city string. We should call it with city "New York".', 'tool_calls': [{'id': 'fc_8fe30358-dcb6-431a-b40c-2569f2bcc820', 'function': {'arguments': '{"city":"New York"}', 'name': 'search_hotel'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 130, 'total_tokens': 193, 'completion_time': 0.131200276, 'completion_tokens_details': {'reasoning_tokens': 34}, 'prompt_time': 0.00494398, 'prompt_tokens_details': None, 'queue_time': 0.05453062, 'total_time': 0.136144256}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4a19b1544c', 'service_tier': 'on_demand', 'finish_reason': 'tool_call

In [23]:
# fraction of tokens

# low token fraction for testing
agent_for_fractionTesting = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[search_hotel],
    checkpointer=InMemorySaver(),
    middleware=[SummarizationMiddleware(
        model="groq:openai/gpt-oss-120b",
        trigger=("fraction",0.005),
        keep=("fraction",0.002)
    )]
)

config_for_fraction = {"configurable":{"thread_id": "test-3"}}

for city in cities:
    print(f"\nHotel search for {city}")
    response = agent_for_fractionTesting.invoke(
        {"messages":[HumanMessage(content=f"Find hotels in {city}")]},
        config=config_for_fraction)

    token = token_counter(response['messages'])
    fraction = token / 128000
    print(f"{city}: ~{token} tokens, ({fraction:.4%}), {len(response['messages'])} msgs") 
    print(f"{response['messages']}")


Hotel search for New York
New York: ~1601 tokens, (1.2508%), 4 msgs
[HumanMessage(content='Find hotels in New York', additional_kwargs={}, response_metadata={}, id='96614f08-8939-408b-82f2-98f89f626f28'), AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants to find hotels in New York. We can call the search_hotel function with city "New York".', 'tool_calls': [{'id': 'fc_f1030821-1108-4ddf-9f6b-59f403ba1f66', 'function': {'arguments': '{"city":"New York"}', 'name': 'search_hotel'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 130, 'total_tokens': 184, 'completion_time': 0.113107439, 'completion_tokens_details': {'reasoning_tokens': 25}, 'prompt_time': 0.004982665, 'prompt_tokens_details': None, 'queue_time': 0.048812714, 'total_time': 0.118090104}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4a19b1544c', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model

In [24]:
def read_email_tool(email_id:str) -> str:
    """Mock function to read an email by its ID""" 
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient:str, subject:str, body:str) -> str:
    """Mock function to send an email""" 
    return f"Email sent to {recipient} with subject {subject} and body {body}"

In [64]:
from langchain.agents.middleware.human_in_the_loop import HumanInTheLoopMiddleware

agent_for_HILtesting = create_agent(
    model="gpt-4o",
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
        interrupt_on={
            "send_email_tool":{ "allowed_decisions":["approve", "reject", "edit"] },
            "read_email_tool":False
        }
    )]
)

In [65]:
config_for_HIL = {"configurable":{"thread_id":"test-approve"}}

# Step 1 : Request
result = agent_for_HILtesting.invoke(
    {"messages":[HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'Hi, how are you?'" )]}, 
    config=config_for_HIL
)

print(result)


{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'Hi, how are you?'", additional_kwargs={}, response_metadata={}, id='75fb48eb-85d0-404e-acb9-8e2272ca795e'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 99, 'total_tokens': 129, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_deacdd5f6f', 'id': 'chatcmpl-CufpziCx7FqD8vDlLcdrp0bep9bhm', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b8e8d-f509-7292-8c10-a7b50790e96f-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'Hello', 'body': 'Hi, how are you?'}, 'id': 'call_loiLXFHurw

In [66]:
# Step 2: Approve
from langgraph.types import Command

if "__interrupt__" in result:
    print(result["__interrupt__"]) 

    result = agent_for_HILtesting.invoke(
        Command(
            resume={
            "decisions":[
                {
                    "type":"approve"
                }
            ]
          }
        ),
        config=config_for_HIL
    )

    print(result)
    print(f"result: {result['messages'][-1].content}")

[Interrupt(value={'action_requests': [{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'Hello', 'body': 'Hi, how are you?'}, 'description': "Tool execution requires approval\n\nTool: send_email_tool\nArgs: {'recipient': 'john@test.com', 'subject': 'Hello', 'body': 'Hi, how are you?'}"}], 'review_configs': [{'action_name': 'send_email_tool', 'allowed_decisions': ['approve', 'reject', 'edit']}]}, id='b5c066a7789bd298e05932b69af10b14')]
{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'Hi, how are you?'", additional_kwargs={}, response_metadata={}, id='75fb48eb-85d0-404e-acb9-8e2272ca795e'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 99, 'total_tokens': 129, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'aud

In [67]:
# reject

config_for_HIL = {"configurable":{"thread_id":"test-reject"}}

# Step 1 : Request
result = agent_for_HILtesting.invoke(
    {"messages":[HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'Hi, how are you?'" )]}, 
    config=config_for_HIL
)

print(result)

# Step 2: Reject
from langgraph.types import Command

if "__interrupt__" in result:
    print(result["__interrupt__"]) 

    result = agent_for_HILtesting.invoke(
        Command(
            resume={
            "decisions":[
                {
                    "type":"reject"
                }
            ]
          }
        ),
        config=config_for_HIL
    )

    print(result)
    print(f"result: {result['messages'][-1].content}")

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'Hi, how are you?'", additional_kwargs={}, response_metadata={}, id='c78507ba-8443-45bc-9776-e4917d726faa'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 99, 'total_tokens': 129, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_deacdd5f6f', 'id': 'chatcmpl-Cufrkg2vxMImeipxME4YiSI9VW7dn', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b8e8f-9f69-77b3-ad2a-b9122c13bdd2-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'Hello', 'body': 'Hi, how are you?'}, 'id': 'call_sfokUzadqW

In [ ]:
# edit

config_for_HIL = {"configurable":{"thread_id":"test-edit"}}

# Step 1 : Request
result = agent_for_HILtesting.invoke(
    {"messages":[HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'Hi, how are you?'" )]}, 
    config=config_for_HIL
)

print(result)

# Step 2: Edit
from langgraph.types import Command

if "__interrupt__" in result:
    print(result["__interrupt__"]) 

    result = agent_for_HILtesting.invoke(
        Command(
            resume={
            "decisions":[
                {
                    "type":"edit",
                    "edited_action":{
                        "name":"send_email_tool",
                        "args":{
                            "recipient":"correct@email.com",
                            "subject":"corrected subject",
                            "body":"corrected body by human"
                        }
                    }
                }
            ]
          }
        ),
        config=config_for_HIL
    )

    print(result)
    print(f"result: {result['messages'][-2].content}")

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'Hi, how are you?'", additional_kwargs={}, response_metadata={}, id='9dbc155c-1804-4230-92f5-67d052ce4248'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 99, 'total_tokens': 129, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_deacdd5f6f', 'id': 'chatcmpl-CugEOLN4Qf51ciTjYHl2K9icofiyf', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019b8ea5-0d25-7851-8523-df0265f5212d-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'correct@email.com', 'subject': 'Correct subject', 'body': 'Correct body'}, 'id': 'call_

In [8]:
# defining state via middleware

from langgraph.runtime import Runtime
from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage,HumanMessage
from langchain.agents import AgentState
from langchain.agents.middleware import AgentMiddleware
from typing import Any
from langchain.agents import create_agent

import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

class CustomState(AgentState):
    """Custom state of the agent."""
    user_preference:dict
    api_call_count:int

class TrackingMiddleware(AgentMiddleware):
    state_schema = CustomState
    name = "tracking_middleware"

    def before_model(self,state:CustomState, runtime):
        """Hook to modify state before a model call"""
        print(f"[TrackingMiddleware] Processing request for use: {state['user_preference']}")

    def after_model(self,state:CustomState,runtime:Runtime) ->None:
        """Hook to update state after a model call."""
        state['api_call_count'] += 1
        print(f"[TrackingMiddleware] Model call count: {state['api_call_count']}")
        return None # must return the response

llm_model = ChatOpenAI(model="gpt-4o")

custom_agent = create_agent(
    model=llm_model,
    tools=[],
    middleware=[TrackingMiddleware()]
)

# when invoking the agent, you must provide initial values for your custom state fields
initial_state={
    "messages":[HumanMessage(content="Hello there!")],
    "user_preference":{"style":"technical"},
    "api_call_count":0
}

result = custom_agent.invoke(initial_state,config={"configurable":{"thread_id":"conv456"}})
print(result)
    



[TrackingMiddleware] Processing request for use: {'style': 'technical'}
[TrackingMiddleware] Model call count: 1
{'messages': [HumanMessage(content='Hello there!', additional_kwargs={}, response_metadata={}, id='50a37ee4-92b3-49f1-9c9c-790d2483afdc'), AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 10, 'total_tokens': 19, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_deacdd5f6f', 'id': 'chatcmpl-D1cgwZZhGMntHYTgeBDLy9cuWFzTH', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019bf145-9f57-72a0-b15a-507c75bca37e-0', usage_metadata={'input_tokens': 10, 'output_tokens': 9, 'total_tokens': 19,